# Evaluación Parcial 1 - Machine Learning
**Asignatura:** MLY0100 Machine Learning  
**Integrantes:** Christian Sandoval, Nicolás Vega  
**Fecha:** 25 de septiembre de 2026

# Fase 1 - Comprensión del Negocio

En este trabajo usamos un dataset de propiedades en venta de Argentina. La idea es revisar los datos, entenderlos y dejarlos preparados para poder usarlos más adelante en Machine Learning.

**Pregunta de negocio:** ¿Qué características de las propiedades, como la ubicación, la superficie y el tipo de propiedad, se relacionan con las diferencias de precio?

**Supuestos:**
- Los valores nulos no significan cero.
- Los valores muy altos se revisan antes de modificarlos.
- Las conclusiones corresponden solamente a este dataset.

**Target para regresión:** `price`, porque es una variable numérica continua.

**Target para clasificación:** `property_type`, porque contiene categorías como Departamento, Casa y PH.

En esta entrega no se entrenan modelos, solo se dejan definidos los targets y se preparan los datos.


# Fase 2 - Comprensión de los Datos

## Carga e inspección

Primero importamos las librerías que vamos a usar. Después cargamos el CSV que viene dentro del archivo ZIP y revisamos sus dimensiones, tipos de datos y una descripción general.


In [ ]:
%matplotlib inline

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

# Nombre del archivo entregado
nombre_zip = 'Evaluación Parcial 1 - Dataset.zip'

# Buscamos el ZIP en la carpeta actual o en la carpeta anterior
if os.path.exists(nombre_zip):
    ruta_zip = nombre_zip
else:
    ruta_zip = '../' + nombre_zip

# Si estamos en Colab y no está el ZIP, lo subimos
if not os.path.exists(ruta_zip):
    from google.colab import files
    files.upload()
    ruta_zip = nombre_zip

# Abrimos el ZIP y cargamos el CSV
with zipfile.ZipFile(ruta_zip) as archivo_zip:
    nombre_csv = [
        nombre for nombre in archivo_zip.namelist()
        if nombre.endswith('DS1-18-Datos-Properati.csv')
        and not nombre.startswith('__MACOSX')
    ][0]

    with archivo_zip.open(nombre_csv) as archivo_csv:
        df = pd.read_csv(archivo_csv)

print('Dimensiones:', df.shape)
print(df.dtypes)
df.info()

display(df.head())
display(df.describe(include='all').T)


El dataset tiene **146.660 filas y 19 columnas**. Al revisar los tipos de datos vemos que las fechas están guardadas como texto. También aparecen valores nulos en `lat`, `lon`, `bathrooms`, `surface_total` y `surface_covered`.

## Estadísticos descriptivos

Ahora calculamos media, mediana, moda, desviación estándar, varianza e IQR de las variables numéricas que consideramos más importantes para el análisis.


In [ ]:
variables = [
    'rooms',
    'bedrooms',
    'bathrooms',
    'surface_total',
    'surface_covered',
    'price'
]

estadisticos = pd.DataFrame({
    'media': df[variables].mean(),
    'mediana': df[variables].median(),
    'moda': df[variables].mode().iloc[0],
    'std': df[variables].std(),
    'varianza': df[variables].var(),
    'Q1': df[variables].quantile(0.25),
    'Q3': df[variables].quantile(0.75)
})

estadisticos['IQR'] = estadisticos['Q3'] - estadisticos['Q1']

estadisticos.round(2)


En `price`, `surface_total` y `surface_covered` la media y la mediana son bastante distintas. Esto nos muestra que las distribuciones no son simétricas y que existen valores extremos.

## Distribuciones

Para mirar mejor los datos hacemos un histograma, boxplots, un gráfico de barras, un scatter y un heatmap. En algunos gráficos usamos el percentil 99 solamente para que los valores muy grandes no deformen la visualización.


In [ ]:
p99_price = df['price'].quantile(0.99)
p99_superficie = df['surface_total'].quantile(0.99)

precios = df.loc[df['price'] <= p99_price, 'price'].dropna()

# Histograma de precio
plt.figure(figsize=(8, 5))
plt.hist(precios, bins=40, edgecolor='black')
plt.axvline(precios.mean(), linestyle='--', label='Media')
plt.axvline(precios.median(), linestyle='-', label='Mediana')
plt.title('Distribución del precio')
plt.xlabel('Precio (USD)')
plt.ylabel('Cantidad')
plt.legend()
plt.show()

# Boxplots
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].boxplot(precios, vert=False)
ax[0].set_title('Boxplot de price')
ax[0].set_xlabel('Precio (USD)')
ax[0].set_ylabel('Distribución')

superficies = df.loc[
    df['surface_total'] <= p99_superficie,
    'surface_total'
].dropna()

ax[1].boxplot(superficies, vert=False)
ax[1].set_title('Boxplot de surface_total')
ax[1].set_xlabel('Superficie total (m²)')
ax[1].set_ylabel('Distribución')

plt.tight_layout()
plt.show()

# Tipos de propiedad con más registros
tipos = df['property_type'].value_counts().head(10)

plt.figure(figsize=(9, 5))
plt.barh(tipos.index, tipos.values)
plt.title('10 tipos de propiedad con más publicaciones')
plt.xlabel('Cantidad')
plt.ylabel('Tipo de propiedad')
plt.gca().invert_yaxis()
plt.show()

# Relación entre superficie y precio
muestra = df[
    (df['surface_total'] <= p99_superficie) &
    (df['price'] <= p99_price)
][['surface_total', 'price']].dropna()

muestra = muestra.sample(5000, random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(muestra['surface_total'], muestra['price'], alpha=0.35)
plt.title('Superficie total y precio')
plt.xlabel('Superficie total (m²)')
plt.ylabel('Precio (USD)')
plt.show()

# Correlación
plt.figure(figsize=(9, 6))
sns.heatmap(df[variables].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlación')
plt.xlabel('Variables')
plt.ylabel('Variables')
plt.show()


Al mirar los gráficos vemos que el precio tiene una distribución asimétrica. También se observa que, en general, cuando aumenta la superficie total el precio tiende a subir, aunque hay bastante dispersión. El tipo de propiedad con más registros es `Departamento`.

# Fase 3 - Preparación de los Datos

## Missing values

Primero revisamos cuántos valores faltantes hay. Como los porcentajes cambian según el tipo de propiedad y `lat` y `lon` muchas veces faltan juntos, para este trabajo los consideramos principalmente como **MAR**. Esto es una interpretación a partir de lo que vemos en los datos.

También probamos `KNNImputer`, porque fue una de las técnicas trabajadas en clases.


In [ ]:
# Cantidad y porcentaje de nulos
resumen_nulos = pd.DataFrame({
    'cantidad_nulos': df.isna().sum(),
    'porcentaje_nulos': (df.isna().mean() * 100).round(2)
})

resumen_nulos = resumen_nulos[
    resumen_nulos['cantidad_nulos'] > 0
].sort_values('porcentaje_nulos', ascending=False)

display(resumen_nulos)

plt.figure(figsize=(8, 5))
plt.barh(resumen_nulos.index, resumen_nulos['porcentaje_nulos'])
plt.title('Porcentaje de valores faltantes')
plt.xlabel('Porcentaje (%)')
plt.ylabel('Variable')
plt.gca().invert_yaxis()
plt.show()

# Probamos KNNImputer con una muestra para que no demore tanto
muestra_knn = df[
    ['bathrooms', 'surface_total', 'surface_covered']
].sample(1500, random_state=42)

datos_knn = muestra_knn.copy()

knn_imputer = KNNImputer(
    n_neighbors=2,
    weights='uniform'
)

datos_knn[
    ['bathrooms', 'surface_total', 'surface_covered']
] = knn_imputer.fit_transform(
    datos_knn[['bathrooms', 'surface_total', 'surface_covered']]
)

comparacion_knn = pd.DataFrame({
    'media_original': muestra_knn.mean(),
    'media_knn': datos_knn.mean(),
    'std_original': muestra_knn.std(),
    'std_knn': datos_knn.std()
})

display(comparacion_knn.round(2))

# Para la limpieza final usamos medianas por grupo
df_limpio = df.copy()

for columna in ['surface_total', 'surface_covered', 'bathrooms']:
    mediana = df_limpio.groupby(
        'property_type',
        observed=False
    )[columna].transform('median')

    df_limpio[columna] = df_limpio[columna].fillna(mediana)

for columna in ['lat', 'lon']:
    mediana = df_limpio.groupby(
        'l3',
        observed=False
    )[columna].transform('median')

    df_limpio[columna] = df_limpio[columna].fillna(mediana)

print('Nulos restantes:', int(df_limpio.isna().sum().sum()))
print('Filas:', len(df_limpio))


La prueba con KNN permitió completar los datos faltantes, pero para la limpieza final elegimos la **mediana agrupada**. La usamos porque es fácil de interpretar y la mediana se ve menos afectada por valores extremos. Después de este paso seguimos con **146.660 filas y 0 nulos**.

## Outliers

Para detectar valores atípicos usamos IQR y Z-score. No eliminamos directamente todos los casos encontrados, porque en este dataset una propiedad con un precio o una superficie muy alta puede ser un dato real.


In [ ]:
# Detección de outliers con IQR y Z-score
lista_outliers = []

for columna in variables:
    q1 = df_limpio[columna].quantile(0.25)
    q3 = df_limpio[columna].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    mascara_iqr = (
        (df_limpio[columna] < limite_inferior) |
        (df_limpio[columna] > limite_superior)
    )

    z_score = np.abs(stats.zscore(df_limpio[columna]))

    lista_outliers.append([
        columna,
        int(mascara_iqr.sum()),
        round(mascara_iqr.mean() * 100, 2),
        int((z_score > 3).sum()),
        round((z_score > 3).mean() * 100, 2)
    ])

resumen_outliers = pd.DataFrame(
    lista_outliers,
    columns=[
        'variable',
        'outliers_iqr',
        'porcentaje_iqr',
        'outliers_zscore',
        'porcentaje_zscore'
    ]
)

display(resumen_outliers)

# Boxplots antes del tratamiento
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].boxplot(df_limpio['price'], vert=False)
ax[0].set_title('Boxplot de price')
ax[0].set_xlabel('Precio (USD)')
ax[0].set_ylabel('Distribución')

ax[1].boxplot(df_limpio['surface_total'], vert=False)
ax[1].set_title('Boxplot de surface_total')
ax[1].set_xlabel('Superficie total (m²)')
ax[1].set_ylabel('Distribución')

plt.tight_layout()
plt.show()

# Limitamos solamente los valores superiores más extremos
tratamiento = []

for columna in variables:
    limite_p99 = df_limpio[columna].quantile(0.99)
    cantidad = int((df_limpio[columna] > limite_p99).sum())
    maximo_antes = df_limpio[columna].max()

    df_limpio[columna] = df_limpio[columna].clip(
        upper=limite_p99
    )

    tratamiento.append([
        columna,
        limite_p99,
        cantidad,
        maximo_antes,
        df_limpio[columna].max()
    ])

resumen_tratamiento = pd.DataFrame(
    tratamiento,
    columns=[
        'variable',
        'limite_p99',
        'ajustados',
        'max_antes',
        'max_despues'
    ]
)

display(resumen_tratamiento.round(2))


IQR encuentra más casos que Z-score en las variables más asimétricas. Para no perder registros usamos el **percentil 99** como límite superior y así reducimos solamente el efecto de los valores más extremos.

## Normalización y estandarización

Después del tratamiento revisamos la asimetría de las variables. Usamos **StandardScaler** para `rooms` y `bedrooms`, que son las menos asimétricas. Para `bathrooms`, superficies y `price` usamos **MinMaxScaler**, porque siguen siendo más asimétricas y así quedan entre 0 y 1.


In [ ]:
# Revisamos la asimetría después de tratar los outliers
print(df_limpio[variables].skew().round(3))

columnas_standard = ['rooms', 'bedrooms']
columnas_minmax = [
    'bathrooms',
    'surface_total',
    'surface_covered',
    'price'
]

df_escalado = df_limpio.copy()

# Estandarización
scaler_standard = StandardScaler()
datos_standard = scaler_standard.fit_transform(
    df_limpio[columnas_standard]
)

df_escalado[
    [columna + '_std' for columna in columnas_standard]
] = datos_standard

# Normalización
scaler_minmax = MinMaxScaler()
datos_minmax = scaler_minmax.fit_transform(
    df_limpio[columnas_minmax]
)

df_escalado[
    [columna + '_minmax' for columna in columnas_minmax]
] = datos_minmax

print('Antes:')
display(
    df_limpio[variables]
    .agg(['mean', 'std', 'min', 'max'])
    .T
    .round(3)
)

print('Después de StandardScaler:')
display(
    df_escalado[
        [columna + '_std' for columna in columnas_standard]
    ]
    .agg(['mean', 'std', 'min', 'max'])
    .T
    .round(3)
)

print('Después de MinMaxScaler:')
display(
    df_escalado[
        [columna + '_minmax' for columna in columnas_minmax]
    ]
    .agg(['mean', 'std', 'min', 'max'])
    .T
    .round(3)
)


Después de aplicar `StandardScaler`, las columnas quedan con media cercana a 0 y desviación estándar cercana a 1. Con `MinMaxScaler`, los valores quedan entre 0 y 1.

## Dataset final y exportación

Para terminar corregimos los tipos de datos de las fechas y variables categóricas. Después revisamos las dimensiones finales y exportamos el dataset limpio a CSV.


In [ ]:
df_final = df_escalado.copy()

# Fechas
for columna in ['start_date', 'end_date', 'created_on']:
    df_final[columna] = df_final[columna].astype(
        'datetime64[s]'
    )

# Variables categóricas
for columna in [
    'l1',
    'l2',
    'l3',
    'currency',
    'property_type',
    'operation_type'
]:
    df_final[columna] = df_final[columna].astype('category')

print('Dimensiones finales:', df_final.shape)
print('Valores nulos:', int(df_final.isna().sum().sum()))
print(df_final.dtypes)

df_final.to_csv(
    'DS1-18-Datos-Properati-preparado.csv',
    index=False
)

print('Dataset exportado correctamente.')


# Conclusiones

En este trabajo completamos las tres primeras fases de CRISP-DM: comprensión del negocio, comprensión de los datos y preparación de los datos.

Al revisar los gráficos vimos que la superficie total tiene una relación positiva con el precio, pero también hay bastante dispersión. Esto indica que el precio no depende solamente de la superficie.

Encontramos datos faltantes en superficies, coordenadas y baños. Probamos `KNNImputer`, pero finalmente usamos medianas agrupadas para completar los nulos sin eliminar filas.

También revisamos los outliers con IQR y Z-score. Como algunos valores altos pueden ser propiedades reales, no eliminamos todas esas filas y usamos el percentil 99 para controlar los casos más extremos.

Finalmente aplicamos `StandardScaler` y `MinMaxScaler` según la distribución observada y corregimos los tipos de datos. El dataset final queda con **146.660 filas, 25 columnas y 0 valores nulos**, preparado para continuar después con el modelado.
